In [0]:

# ===== PARÁMETROS: cambiá solo esto para re-ejecutar con otra ciudad o fechas =====
CIUDAD = "Buenos Aires"
ZONA_HORARIA = "America/Argentina/Buenos_Aires"
FECHA_DESDE = "2026-03-01"
FECHA_HASTA = "2026-05-30"
RUTA_VIAJES = "/Volumes/workspace/nexourbano/nexo/viajes.csv"
TABLA_SILVER = "workspace.nexourbano.silver_clima_viajes"
HASHTAG_SOCIAL = "scooter"
TABLA_CLIMA = "workspace.nexourbano.silver_clima_horario"
TABLA_SOCIAL = "workspace.nexourbano.silver_social_menciones"

In [0]:
# 1. Leer el CSV de viajes (mismo millón de viajes que perfilamos en M1)
import pandas as pd

print(f"Ciudad: {CIUDAD} | Período: {FECHA_DESDE} a {FECHA_HASTA}")
viajes = pd.read_csv(RUTA_VIAJES)

# 2. Forma del dataset y nulos por columna
print("Filas y columnas:", viajes.shape)
print("\nNulos por columna:")
print(viajes.isnull().sum())

# 3. Primeras filas
display(viajes.head())

Ciudad: Buenos Aires | Período: 2026-03-01 a 2026-05-30
Filas y columnas: (1000000, 17)

Nulos por columna:
ride_id          0
ts_start         0
ts_end           0
duration_s       0
rider_id         0
vehicle_id       0
station_start    0
station_end      0
barrio_start     0
barrio_end       0
lat_start        0
lon_start        0
lat_end          0
lon_end          0
km               0
fare_ars         0
clima_codigo     0
dtype: int64


ride_id,ts_start,ts_end,duration_s,rider_id,vehicle_id,station_start,station_end,barrio_start,barrio_end,lat_start,lon_start,lat_end,lon_end,km,fare_ars,clima_codigo
T0000000,2026-05-26T10:46:00+00:00,2026-05-26T10:49:15+00:00,195,R0002,V0052,ST013,ST011,Recoleta,Chacarita,-34.613389,-58.345158,-34.581593,-58.408352,2.071,231.1,80
T0000001,2026-05-25T05:28:00+00:00,2026-05-25T05:30:57+00:00,177,R0020,V0060,ST004,ST003,Caballito,Belgrano,-34.589067,-58.344321,-34.586905,-58.403828,2.177,229.04,80
T0000002,2026-03-04T07:02:00+00:00,2026-03-04T07:08:31+00:00,391,R0084,V0050,ST018,ST016,Villa Crespo,Caballito,-34.607356,-58.376201,-34.573486,-58.42079,1.465,275.44,2
T0000003,2026-03-22T22:01:00+00:00,2026-03-22T22:06:11+00:00,311,R0068,V0044,ST019,ST020,Nunez,Colegiales,-34.59347,-58.355105,-34.63832,-58.41055,3.82,341.63,1
T0000004,2026-03-26T09:02:00+00:00,2026-03-26T09:07:34+00:00,334,R0022,V0019,ST005,ST006,Almagro,Villa Crespo,-34.583829,-58.345873,-34.639631,-58.408175,0.129,202.07,0


In [0]:
# 1. Pasar la fecha de inicio a hora de Buenos Aires (en el archivo está en UTC)
viajes["ts_start"] = pd.to_datetime(viajes["ts_start"])
viajes["hora_local"] = viajes["ts_start"].dt.tz_convert(ZONA_HORARIA).dt.hour

# 2. Métrica 1: viajes por hora del día (las 3 horas con más viajes)
por_hora = viajes.groupby("hora_local").size().sort_values(ascending=False)
print("1. Horas con más viajes:")
print(por_hora.head(3))

# 3. Métricas 2 y 3: duración mediana (p50) y p95, en minutos
p50 = viajes["duration_s"].quantile(0.50) / 60
p95 = viajes["duration_s"].quantile(0.95) / 60
print(f"\n2. Duración p50: {p50:.1f} minutos")
print(f"3. Duración p95: {p95:.1f} minutos")

# 4. Métrica 4: las 5 estaciones desde donde más viajes salen
print("\n4. Top 5 estaciones de origen:")
print(viajes["station_start"].value_counts().head(5))

# 5. Métrica 5: ingreso medio por viaje
print(f"\n5. Ingreso medio por viaje: {viajes['fare_ars'].mean():.2f} ARS")

1. Horas con más viajes:
hora_local
8     42012
17    41908
10    41862
dtype: int64

2. Duración p50: 4.0 minutos
3. Duración p95: 13.5 minutos

4. Top 5 estaciones de origen:
station_start
ST014    42068
ST002    41977
ST018    41933
ST005    41863
ST012    41862
Name: count, dtype: int64

5. Ingreso medio por viaje: 397.85 ARS


In [0]:
# 1. Elegir las 3 estaciones con más viajes de salida
top3 = viajes["station_start"].value_counts().head(3).index

# 2. Quedarnos solo con los viajes que salen de esas 3 estaciones
viajes_top3 = viajes[viajes["station_start"].isin(top3)]

# 3. Ubicación de cada estación = promedio de lat/lon de sus viajes de salida
estaciones = viajes_top3.groupby("station_start")[["lat_start", "lon_start"]].mean().round(4)
estaciones = estaciones.reset_index()
display(estaciones)

station_start,lat_start,lon_start
ST002,-34.6159,-58.3741
ST014,-34.5665,-58.3822
ST018,-34.6074,-58.3762


In [0]:
# 1. Pedir a Open-Meteo el clima hora por hora de cada estación (historial, en UTC)
import json
import urllib.request

tablas = []   # acá juntamos el clima de cada estación
for i, est in estaciones.iterrows():
    url = ("https://archive-api.open-meteo.com/v1/archive"
           f"?latitude={est['lat_start']}&longitude={est['lon_start']}"
           f"&start_date={FECHA_DESDE}&end_date={FECHA_HASTA}"
           "&hourly=temperature_2m,precipitation,wind_speed_10m,weather_code")
    respuesta = json.loads(urllib.request.urlopen(url, timeout=60).read())
    tabla = pd.DataFrame(respuesta["hourly"])
    tabla["station_id"] = est["station_start"]    # de qué estación es
    tablas.append(tabla)

# 2. Juntar las tablas y ponerles los nombres del contrato (silver_clima.yaml)
clima = pd.concat(tablas)
clima = clima.rename(columns={"time": "ts_hora", "temperature_2m": "temp_c",
                              "precipitation": "precip_mm", "wind_speed_10m": "wind_kmh"})
clima["ts_hora"] = pd.to_datetime(clima["ts_hora"]).dt.tz_localize("UTC")

print("Filas y columnas del clima:", clima.shape)
display(clima.head())

Filas y columnas del clima: (6552, 6)


ts_hora,temp_c,precip_mm,wind_kmh,weather_code,station_id
2026-03-01T00:00:00.000Z,25.4,0.0,6.5,0,ST002
2026-03-01T01:00:00.000Z,25.3,0.0,11.6,0,ST002
2026-03-01T02:00:00.000Z,24.8,0.0,11.0,0,ST002
2026-03-01T03:00:00.000Z,24.5,0.0,10.7,0,ST002
2026-03-01T04:00:00.000Z,24.2,0.0,10.3,0,ST002


In [0]:
# Reglas de calidad del contrato silver_clima.yaml

# C1: weather_code tiene que estar en la lista de códigos de clima válidos (tabla WMO)
codigos_validos = [0, 1, 2, 3, 45, 48, 51, 53, 55, 56, 57, 61, 63, 65, 66, 67,
                   71, 73, 75, 77, 80, 81, 82, 85, 86, 95, 96, 99]
c1_ok = clima["weather_code"].isin(codigos_validos)

# C2: la temperatura tiene que estar entre -10 y 50 grados
c2_ok = clima["temp_c"].between(-10, 50)

print(f"C1 rechazados (código de clima inválido): {(~c1_ok).sum()}")
print(f"C2 rechazados (temperatura fuera de rango): {(~c2_ok).sum()}")

# Nos quedamos solo con las filas que cumplen las dos reglas
clima_ok = clima[c1_ok & c2_ok]
print(f"Clima válido: {len(clima_ok)} de {len(clima)} filas")

C1 rechazados (código de clima inválido): 0
C2 rechazados (temperatura fuera de rango): 0
Clima válido: 6552 de 6552 filas


In [0]:
# Feed social público: Mastodon, posts con un hashtag (no requiere API key)
import re

url = f"https://mastodon.social/api/v1/timelines/tag/{HASHTAG_SOCIAL}?limit=40"
pedido = urllib.request.Request(url, headers={"User-Agent": "nexourbano-tp"})
posts = json.loads(urllib.request.urlopen(pedido, timeout=30).read())

# Armar la tabla con las columnas del contrato silver_social.yaml
social = pd.DataFrame({
    "post_uri": [p["uri"] for p in posts],
    "created_at": [p["created_at"] for p in posts],
    "text": [re.sub("<[^>]+>", "", p["content"]) for p in posts],   # sacamos las etiquetas HTML
    "plataforma": "mastodon",
})

# Reglas de calidad: P1 post_uri único, P2 texto no vacío
social = social.drop_duplicates("post_uri")
social = social[social["text"].str.strip() != ""]

print(f"Posts válidos: {len(social)}")
print(f"Fechas: de {social['created_at'].min()} a {social['created_at'].max()}")
display(social.head())

Posts válidos: 40
Fechas: de 2026-09-24T17:59:22.000Z a 2026-10-16T15:00:01.741Z


post_uri,created_at,text,plataforma
https://bbs.kawa-kun.com/objects/748a4da6-9e4c-4752-8c50-52083734362f,2026-10-16T15:00:01.741Z,#motorcycle #scooter #meLink: https://instagram.com/p/CQ1HxMAlwz6,mastodon
https://fedihub.space/notes/ase66loajk9z001u,2026-10-16T06:47:38.554Z,"Kritikpunkt entdeckt beim Scooter - Die AppEcooter von Niu und die Companion App, es gibt doch was zu bemängeln.https://gehirn-blogging.de/blog/kritikpunkt-entdeckt-beim-scooter-die-app#scooter #emobilityfolgt meinem Blog für mehr. :-)",mastodon
https://mastodon.social/users/fuerthaktuell/statuses/117448644039436449,2026-10-16T04:14:45.080Z,E-Scooter in Fürth: 160 feste Parkplätze und neue Abstellregelnhttps://go.fuerthaktuell.de/12937#fürth #südstadt #innenstadt #e #scooter #escooter #scooterparken #parkplätze #abstellregeln #sammelparkplätze #verkehrssicherheit #öffentlicherraum #radverkehr #fußverkehr #franken #mittelfranken #bayern #news,mastodon
https://mastodon.social/users/NieuwsJunkies/statuses/117441141701426640,2026-10-14T20:26:48.484Z,"📰 Gevangene ontsnapt bij doktersafspraak in Duitsland, bewakers vastgebonden achtergelatenhttps://nieuwsjunkies.nl/artikel/1Ueu🕥 22:23 | RTL Nieuws🔸 #Ontsnapping #Scooter #Gevangenis #Duitsland #Dokter",mastodon
https://bbs.kawa-kun.com/objects/a1c74b07-0aae-472e-9b69-bc6d56288651,2026-10-14T15:00:02.267Z,Getting a #motorcycle / #scooter really has opened up some of my old cycling routes for me. I'm able to get shots that I normally wouldn't have gotten on the bicycle due to time constraints. #WhidbeyIsland #KitsapPeninsula #PortTownsendLink: https://instagram.com/p/CQeCVTrhHaA,mastodon


In [0]:
# Reglas del contrato bronze_viajes.yaml

# La fecha de fin viene como texto: la pasamos a fecha
ts_fin = pd.to_datetime(viajes["ts_end"])

# R1: la duración tiene que estar entre 30 segundos y 4 horas (14400 s)
r1_ok = viajes["duration_s"].between(30, 14400)

# R2: la distancia tiene que ser mayor a 0
r2_ok = viajes["km"] > 0

# R3: el viaje tiene que terminar después de empezar
r3_ok = ts_fin > viajes["ts_start"]

# R4: el ride_id no puede estar repetido
r4_ok = viajes["ride_id"].duplicated() == False

# R5: rider_id, vehicle_id y station_start no pueden ser nulos
r5_ok = viajes["rider_id"].notnull() & viajes["vehicle_id"].notnull() & viajes["station_start"].notnull()

# Mostrar cuántos viajes rompe cada regla
print("R1 rechazados (duración):", (r1_ok == False).sum())
print("R2 rechazados (km <= 0):", (r2_ok == False).sum())
print("R3 rechazados (fin antes del inicio):", (r3_ok == False).sum())
print("R4 rechazados (ride_id repetido):", (r4_ok == False).sum())
print("R5 rechazados (nulos):", (r5_ok == False).sum())

# R6: velocidad media en km/h. Solo avisamos, NO rechazamos
velocidad = viajes["km"] / (viajes["duration_s"] / 3600)
print("R6 aviso (velocidad > 60 km/h):", (velocidad > 60).sum())

# Separar: los que cumplen TODAS las reglas van a Silver, el resto a cuarentena
todas_ok = r1_ok & r2_ok & r3_ok & r4_ok & r5_ok
silver_viajes = viajes[todas_ok]
cuarentena = viajes[todas_ok == False]

print("\nViajes en Bronze:", len(viajes))
print("Viajes en Silver:", len(silver_viajes))
print("Viajes en cuarentena:", len(cuarentena))
print("Silver + cuarentena = Bronze ?", len(silver_viajes) + len(cuarentena) == len(viajes))

R1 rechazados (duración): 20184
R2 rechazados (km <= 0): 56
R3 rechazados (fin antes del inicio): 0
R4 rechazados (ride_id repetido): 0
R5 rechazados (nulos): 0
R6 aviso (velocidad > 60 km/h): 447298

Viajes en Bronze: 1000000
Viajes en Silver: 979760
Viajes en cuarentena: 20240
Silver + cuarentena = Bronze ? True


In [0]:
# 1. Quedarnos con los viajes válidos que salen de las 3 estaciones con clima
viajes_3 = silver_viajes[silver_viajes["station_start"].isin(top3)].copy()

# 2. Crear la clave de la hora: 10:46 pasa a ser 10:00 (en UTC, igual que el clima)
viajes_3["ts_hora"] = viajes_3["ts_start"].dt.floor("h")

# 3. Ponerle a la estación el mismo nombre que tiene en la tabla de clima
viajes_3 = viajes_3.rename(columns={"station_start": "station_id"})

# 4. Cruzar (JOIN) viajes con clima por estación y hora
silver_clima_viajes = viajes_3.merge(clima_ok, on=["station_id", "ts_hora"], how="left")

# 5. Revisar el resultado
print("Filas y columnas:", silver_clima_viajes.shape)
print("Viajes sin clima:", silver_clima_viajes["weather_code"].isnull().sum())
display(silver_clima_viajes[["ride_id", "station_id", "ts_hora", "duration_s", "temp_c", "precip_mm", "weather_code"]].head())

Filas y columnas: (123479, 23)
Viajes sin clima: 0


ride_id,station_id,ts_hora,duration_s,temp_c,precip_mm,weather_code
T0000002,ST018,2026-03-04T07:00:00.000Z,391,21.6,0.0,3
T0000005,ST018,2026-03-29T12:00:00.000Z,283,22.0,0.1,51
T0000014,ST002,2026-05-15T23:00:00.000Z,222,14.5,0.0,3
T0000037,ST002,2026-05-24T00:00:00.000Z,216,10.8,0.0,0
T0000041,ST018,2026-04-19T04:00:00.000Z,228,20.7,0.0,0


In [0]:
# 1. Pasar la fecha de fin a fecha (venía como texto)
silver_clima_viajes["ts_end"] = pd.to_datetime(silver_clima_viajes["ts_end"])

# 2. Hora local de Buenos Aires, fecha y hora del día
silver_clima_viajes["ts_local"] = silver_clima_viajes["ts_start"].dt.tz_convert(ZONA_HORARIA)
silver_clima_viajes["fecha"] = silver_clima_viajes["ts_local"].dt.date
silver_clima_viajes["hora"] = silver_clima_viajes["ts_local"].dt.hour

# 3. Duración en minutos
silver_clima_viajes["duracion_min"] = silver_clima_viajes["duration_s"] / 60

# 4. Clima adverso: código de clima 51 o más (llovizna, lluvia, chubascos, tormenta)
silver_clima_viajes["clima_adverso"] = silver_clima_viajes["weather_code"] >= 51

# 5. Volver a llamar station_start a la estación, como dice el contrato
silver_clima_viajes = silver_clima_viajes.rename(columns={"station_id": "station_start"})

# 6. Quedarnos solo con las columnas que definimos para Silver
columnas = ["ride_id", "ts_start", "ts_end", "ts_local", "fecha", "hora",
            "duration_s", "duracion_min", "rider_id", "vehicle_id", "station_start",
            "km", "fare_ars", "temp_c", "precip_mm", "weather_code", "clima_adverso"]
silver_clima_viajes = silver_clima_viajes[columnas]

# 7. Revisar
print("Filas y columnas:", silver_clima_viajes.shape)
porcentaje = 100 * silver_clima_viajes["clima_adverso"].mean()
print(f"Viajes con clima adverso: {porcentaje:.1f} %")
display(silver_clima_viajes.head())

Filas y columnas: (123479, 17)
Viajes con clima adverso: 13.3 %


ride_id,ts_start,ts_end,ts_local,fecha,hora,duration_s,duracion_min,rider_id,vehicle_id,station_start,km,fare_ars,temp_c,precip_mm,weather_code,clima_adverso
T0000002,2026-03-04T07:02:00.000Z,2026-03-04T07:08:31.000Z,2026-03-04T07:02:00.000Z,2026-03-04,4,391,6.516666666666667,R0084,V0050,ST018,1.465,275.44,21.6,0.0,3,false
T0000005,2026-03-29T12:29:00.000Z,2026-03-29T12:33:43.000Z,2026-03-29T12:29:00.000Z,2026-03-29,9,283,4.716666666666667,R0020,V0063,ST018,2.827,292.14,22.0,0.1,51,true
T0000014,2026-05-15T23:36:00.000Z,2026-05-15T23:39:42.000Z,2026-05-15T23:36:00.000Z,2026-05-15,20,222,3.7,R0099,V0015,ST002,4.272,328.59,14.5,0.0,3,false
T0000037,2026-05-24T00:06:00.000Z,2026-05-24T00:09:36.000Z,2026-05-24T00:06:00.000Z,2026-05-23,21,216,3.6,R0047,V0065,ST002,5.507,375.87,10.8,0.0,0,false
T0000041,2026-04-19T04:44:00.000Z,2026-04-19T04:47:48.000Z,2026-04-19T04:44:00.000Z,2026-04-19,1,228,3.8,R0029,V0027,ST018,4.718,348.52,20.7,0.0,0,false


In [0]:
# 1. Pasar la tabla de pandas a Spark (el motor de Databricks)
tabla_spark = spark.createDataFrame(silver_clima_viajes)

# 2. Guardarla como tabla Delta en el esquema nexourbano
#    "overwrite" reemplaza la tabla si ya existe: así el notebook se puede re-ejecutar
tabla_spark.write.mode("overwrite").saveAsTable(TABLA_SILVER)

# 3. Comprobar que quedó guardada: contar filas y ver el formato
print("Filas guardadas en", TABLA_SILVER, ":", spark.table(TABLA_SILVER).count())
display(spark.sql(f"DESCRIBE DETAIL {TABLA_SILVER}").select("format", "numFiles", "sizeInBytes"))

Filas guardadas en workspace.nexourbano.silver_clima_viajes : 123479


format,numFiles,sizeInBytes
delta,8,4153596


In [0]:
# 1. Guardar el clima válido (silver_clima.yaml)
spark.createDataFrame(clima_ok).write.mode("overwrite").saveAsTable(TABLA_CLIMA)

# 2. Guardar los posts de Mastodon (silver_social.yaml)
#    created_at venía como texto: lo pasamos a fecha antes de guardar
social["created_at"] = pd.to_datetime(social["created_at"])
spark.createDataFrame(social).write.mode("overwrite").saveAsTable(TABLA_SOCIAL)

# 3. Comprobar cuántas filas tiene cada tabla
print("Clima:", spark.table(TABLA_CLIMA).count(), "filas")
print("Social:", spark.table(TABLA_SOCIAL).count(), "filas")

Clima: 6552 filas
Social: 40 filas
